In [2]:
import os 
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns 

%store -r df_final


In [ ]:

# daily counts / price buckets 

df_eda_v0 = df_final.with_columns(
    pl.from_epoch("window_start_ts").dt.date().alias("market_date"),
    pl.when(pl.col("resolution").is_in([0, 1]))
    .then(pl.lit("known"))
    .otherwise(pl.lit("unknown"))
    .alias("resolution_status"),
)

daily_counts = (
    df_eda_v0.group_by(["market_date", "resolution_status"])
    .len()
    .sort("market_date")
)
price_age_buckets = (
    df_eda_v0
    .with_columns(
        pl.col("price_age_seconds").cut(
            [10, 20, 30, 45, 60, 120],
            labels=["0-10", "11-20", "21-30", "31-45", "46-60", "61-120", "120+"],
        ).alias("price_age_bucket")
    )
    .group_by("price_age_bucket")
    .len()
    .sort("price_age_bucket")
)

In [8]:
# plots for daily counts / price age buckets 

output_dir = ("../reports/figures") 

sns.set_theme(style="whitegrid", palette="muted")

plt.figure(figsize=(12, 6))

sns.barplot(
    data=daily_counts,         
    x="market_date",
    y="len",
    hue="resolution_status"
)

plt.title("Daily Counts by Resolution Status", fontsize=14, pad=15)
plt.xlabel("Market Date", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()               

daily_counts_path = os.path.join(output_dir, "daily_counts_TS.png")
plt.savefig(daily_counts_path, dpi=300)
plt.close()                    


plt.figure(figsize=(10, 5))

sns.barplot(
    data=price_age_buckets,     
    x="price_age_bucket",
    y="len",
    palette="viridis"       
)

plt.title("Volume Distribution by Price Age Buckets", fontsize=14, pad=15)
plt.xlabel("Price Age Bucket", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.tight_layout()


price_age_buckets_path = os.path.join(output_dir, "price_age_buckets_TS.png")
plt.savefig(price_age_buckets_path, dpi=300)
plt.close()


C:\Users\Anigma PC\AppData\Local\Temp\ipykernel_9804\3914612300.py:29: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


In [ ]:

# construct df_eda_v1, define explicit price threshold (note: no price_age_seconds exceeds 10seconds from prediction_ts )

MAX_PRICE_AGE_SECONDS = 60
EXCLUDED_DATE = "2026-04-28"

df_eda_v1 = (
    df_final
    .with_columns(pl.from_epoch("window_start_ts").dt.date().alias("market_date"))
    .filter(
        pl.col("resolution").is_in([0, 1])
        & pl.col("up_price").is_not_null()
        & (pl.col("price_age_seconds") <= MAX_PRICE_AGE_SECONDS)
        & (pl.col("market_date") != pl.date(2026,4,28))
    )
)



In [22]:
import polars.selectors as cs


df_eda_v1.select(cs.by_dtype(pl.String))

market_id,market_slug,crypto,timeframe,resolution_status,winning_outcome_label
str,str,str,str,str,str
"""0xee052e9749e36868aaeea7efb875…","""btc-updown-5m-1770858600""","""BTC""","""5-minute""","""resolved""","""Up"""
"""0xd4bb645f374beb078157da12cd4f…","""btc-updown-5m-1770858900""","""BTC""","""5-minute""","""resolved""","""Down"""
"""0x72b05d06e0a8ccc2a986bcda9a71…","""btc-updown-5m-1770859200""","""BTC""","""5-minute""","""resolved""","""Down"""
"""0x808c50a448b438106958d188f8b3…","""btc-updown-5m-1770859500""","""BTC""","""5-minute""","""resolved""","""Up"""
"""0xe7710c182b8b36312fb7d90bc2fc…","""btc-updown-5m-1770859800""","""BTC""","""5-minute""","""resolved""","""Down"""
…,…,…,…,…,…
"""0x255da457b9c2cbd8e8a09c2dec6a…","""btc-updown-5m-1777332900""","""BTC""","""5-minute""","""resolved""","""Down"""
"""0x155add1d1a7eda2e33940a3fbdec…","""btc-updown-5m-1777333200""","""BTC""","""5-minute""","""resolved""","""Up"""
"""0xd2b24374e1376b6b476d56b6a895…","""btc-updown-5m-1777333500""","""BTC""","""5-minute""","""resolved""","""Up"""
